# 🏅 1.5 užduotis (1): iš anksto apmokyti modeliai su savo nuotraukomis (sporto tema)

Perdarytas dėstytojo notebook'as **„cnn One Image“ (Transfer learning 1)**: vietoj Vilniaus, Kauno ir Klaipėdos nuotraukų naudojamos **sporto nuotraukos**, susijusios su mūsų projektu – sporto treniruočių planuotoju.
Klausimas: ar be jokio papildomo mokymo galima atpažinti, kas nuotraukoje (sporto šaka, įranga, žmonės)? Tai būtų naudinga, pvz., automatiškai nustatant vartotojo turimą įrangą iš nuotraukos.

*[Edvin Gladkich, Nojus Sadaunykas]*

| Nuotrauka | Šaltinis (Wikimedia Commons) | Licencija |
|---|---|---|
| Pritūpimai su štanga | [Woman doing squat workout in gym with barbell, back view.jpg](https://commons.wikimedia.org/wiki/File:Woman_doing_squat_workout_in_gym_with_barbell,_back_view.jpg) | CC BY 2.0 |
| Hanteliai ant stovo | [Fitness dumbbell marked with a green logo resting on a gym rack.jpg](https://commons.wikimedia.org/wiki/File:Fitness_dumbbell_marked_with_a_green_logo_resting_on_a_gym_rack.jpg) | CC BY 2.0 |
| Sporto salė | [New Drawsko Pomorskie Training Area Gym (7411444).jpg](https://commons.wikimedia.org/wiki/File:New_Drawsko_Pomorskie_Training_Area_Gym_(7411444).jpg) | Public domain |
| Svarstis (kettlebell) | [Kettlebell workout.jpg](https://commons.wikimedia.org/wiki/File:Kettlebell_workout.jpg) | CC BY-SA 4.0 |
| Maratonas | [View of London Marathon runners on Lower Thames Street - geograph.org.uk - 7468857.jpg](https://commons.wikimedia.org/wiki/File:View_of_London_Marathon_runners_on_Lower_Thames_Street_-_geograph.org.uk_-_7468857.jpg) | CC BY-SA 2.0 |
| Dviračių lenktynės | [Japan Cup Cycle Road Race 2019 PA200787.jpg](https://commons.wikimedia.org/wiki/File:Japan_Cup_Cycle_Road_Race_2019_PA200787.jpg) | CC BY-SA 4.0 |
| Krepšinis | [Harrison Barnes dunk USA Basketball Blue vs White game Las Vegas 2013.jpg](https://commons.wikimedia.org/wiki/File:Harrison_Barnes_dunk_USA_Basketball_Blue_vs_White_game_Las_Vegas_2013.jpg) | Public domain |
| Futbolas | [Rivers United vs Katsina United match at Adokiye Amiesimaka Stadium during a rainy evening, Port Harcourt 26.jpg](https://commons.wikimedia.org/wiki/File:Rivers_United_vs_Katsina_United_match_at_Adokiye_Amiesimaka_Stadium_during_a_rainy_evening,_Port_Harcourt_26.jpg) | CC0 |
| Plaukimas | [Swimming race at Gilberton(GN00325A).jpg](https://commons.wikimedia.org/wiki/File:Swimming_race_at_Gilberton(GN00325A).jpg) | CC0 |
| Laipiojimo siena | [Indoor Climbing Wall Bouldering Mats.jpg](https://commons.wikimedia.org/wiki/File:Indoor_Climbing_Wall_Bouldering_Mats.jpg) | CC BY 2.0 |
| Tenisas | [Female Tennis Player About to Serve (6736686675).jpg](https://commons.wikimedia.org/wiki/File:Female_Tennis_Player_About_to_Serve_(6736686675).jpg) | CC BY 2.0 |

## Kaip veikia originalus notebook'as (Transfer learning 1)

| Žingsnis | Ką daro originalas | Kodėl |
|---|---|---|
| 1. Pagalbinės funkcijos | Nuotrauka atsisiunčiama iš URL, paverčiama `float32` [0, 1], pridedamas paketo matmuo ir pakeičiamas dydis su `resize_with_pad` (proporcijos išsaugomos). | Modeliai tikisi fiksuoto dydžio [1, h, w, 3] įvesties. |
| 2. Klasifikacija | Iš TF Hub įkeliami du **EfficientNetV2-S** klasifikatoriai: apmokytas su ImageNet-21k (~21 000 klasių) ir ImageNet-21k + papildomai ImageNet-1k (1 000 klasių). Rodomos 4 labiausiai tikėtinos klasės su `softmax` tikimybėmis. | Modeliai jau išmokti milijonais nuotraukų – jų žinios „perkeliamos“ į mūsų uždavinį be mokymo. 21k modelis žino daug smulkesnių klasių, 1k – mažiau, bet patikimesnių. |
| 3. Iškirpimas | Slankikliais pasirenkamas nuotraukos fragmentas, kuris padidinamas ir vėl klasifikuojamas. | Klasifikatorius grąžina vieną atsakymą visai nuotraukai; iškirpus fragmentą galima atpažinti smulkesnius objektus. |
| 4. Objektų aptikimas | **SSD MobileNet v2** (Open Images v4, 600 klasių) randa objektus ir nubrėžia rėmelius su tikimybėmis. | Aptikimas atsako ne tik „kas“, bet ir „kur“, ir randa kelis objektus. |
| 5. Segmentacija | **HRNet** (ADE20K, 150 klasių) kiekvienam pikseliui priskiria klasę; parodomos kaukės (maskės) atskiroms klasėms. | Tiksliausia scenos analizė – žinoma kiekvieno pikselio klasė (žmogus, grindys, siena, dangus…). |

## Ką keičiame
- Savos (sporto) nuotraukos vietoj miestų nuotraukų.
- Segmentacijos kaukės parenkamos automatiškai – keturios didžiausią plotą užimančios klasės (originale indeksai `[2, 5, 13, 18]` įrašyti ranka).
- **Papildomai:** visos 11 nuotraukų apdorojamos automatiškai ir suvedamos į vieną lentelę – kaip kiekvienas modelis „supranta“ sporto scenas.

In [ ]:
#@title 🔗 SETUP: bibliotekos
!pip install -q tensorflow_hub
import math, json, requests
from io import BytesIO
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub
import matplotlib.pyplot as plt
from PIL import Image, ImageColor, ImageDraw, ImageFont
from IPython.display import HTML, display
print("TensorFlow", tf.__version__, "| TF Hub", hub.__version__)

In [ ]:
#@title 🧰 STEP 1: Pagalbinės funkcijos (kaip originale)
HEADERS = {"User-Agent": "TensorflowCourseProject/1.0 (VILNIUS TECH student project)"}  # Wikimedia reikalauja User-Agent

SPORT_PHOTOS = {
    "Pritūpimai su štanga": "https://thumb.wikimedia.org/wikipedia/commons/thumb/f/f9/Woman_doing_squat_workout_in_gym_with_barbell%2C_back_view.jpg/1280px-Woman_doing_squat_workout_in_gym_with_barbell%2C_back_view.jpg",
    "Hanteliai ant stovo": "https://thumb.wikimedia.org/wikipedia/commons/thumb/9/90/Fitness_dumbbell_marked_with_a_green_logo_resting_on_a_gym_rack.jpg/1280px-Fitness_dumbbell_marked_with_a_green_logo_resting_on_a_gym_rack.jpg",
    "Sporto salė": "https://thumb.wikimedia.org/wikipedia/commons/thumb/7/74/New_Drawsko_Pomorskie_Training_Area_Gym_%287411444%29.jpg/1280px-New_Drawsko_Pomorskie_Training_Area_Gym_%287411444%29.jpg",
    "Svarstis (kettlebell)": "https://thumb.wikimedia.org/wikipedia/commons/thumb/3/30/Kettlebell_workout.jpg/1280px-Kettlebell_workout.jpg",
    "Maratonas": "https://thumb.wikimedia.org/wikipedia/commons/thumb/9/92/View_of_London_Marathon_runners_on_Lower_Thames_Street_-_geograph.org.uk_-_7468857.jpg/1280px-View_of_London_Marathon_runners_on_Lower_Thames_Street_-_geograph.org.uk_-_7468857.jpg",
    "Dviračių lenktynės": "https://thumb.wikimedia.org/wikipedia/commons/thumb/d/db/Japan_Cup_Cycle_Road_Race_2019_PA200787.jpg/1280px-Japan_Cup_Cycle_Road_Race_2019_PA200787.jpg",
    "Krepšinis": "https://thumb.wikimedia.org/wikipedia/commons/thumb/6/64/Harrison_Barnes_dunk_USA_Basketball_Blue_vs_White_game_Las_Vegas_2013.jpg/1280px-Harrison_Barnes_dunk_USA_Basketball_Blue_vs_White_game_Las_Vegas_2013.jpg",
    "Futbolas": "https://thumb.wikimedia.org/wikipedia/commons/thumb/4/40/Rivers_United_vs_Katsina_United_match_at_Adokiye_Amiesimaka_Stadium_during_a_rainy_evening%2C_Port_Harcourt_26.jpg/1280px-Rivers_United_vs_Katsina_United_match_at_Adokiye_Amiesimaka_Stadium_during_a_rainy_evening%2C_Port_Harcourt_26.jpg",
    "Plaukimas": "https://thumb.wikimedia.org/wikipedia/commons/thumb/b/bb/Swimming_race_at_Gilberton%28GN00325A%29.jpg/1280px-Swimming_race_at_Gilberton%28GN00325A%29.jpg",
    "Laipiojimo siena": "https://thumb.wikimedia.org/wikipedia/commons/thumb/7/74/Indoor_Climbing_Wall_Bouldering_Mats.jpg/1280px-Indoor_Climbing_Wall_Bouldering_Mats.jpg",
    "Tenisas": "https://thumb.wikimedia.org/wikipedia/commons/thumb/7/77/Female_Tennis_Player_About_to_Serve_%286736686675%29.jpg/1280px-Female_Tennis_Player_About_to_Serve_%286736686675%29.jpg"
}

def load_image_from_url(img_url):
    """Grąžina [1, aukštis, plotis, 3] float32 [0, 1] masyvą."""
    response = requests.get(img_url, headers=HEADERS, timeout=60)
    response.raise_for_status()
    image = np.array(Image.open(BytesIO(response.content)).convert("RGB"))
    return tf.image.convert_image_dtype(image[tf.newaxis, ...], tf.float32)

def load_image(image_url, image_size=256):
    img_raw = load_image_from_url(image_url)
    return tf.image.resize_with_pad(img_raw, image_size, image_size), img_raw

def show_image(image, title=""):
    plt.figure(figsize=(7, 7)); plt.imshow(image[0]); plt.axis("off"); plt.title(title); plt.show()

def top_classes(p_image, p_classifier, p_classes, k=4):
    probabilities = tf.nn.softmax(p_classifier(p_image)).numpy()[0]
    top_k = np.argsort(probabilities)[::-1][:k]
    return [(p_classes[i], float(probabilities[i])) for i in top_k]

def format_answer(answer):
    return "<br>".join(f"{i + 1}) {name}; p = {p:.4f}" for i, (name, p) in enumerate(answer))

In [ ]:
#@title 🧠 STEP 2: Du EfficientNetV2 klasifikatoriai ir jų klasių sąrašai (kaip originale)
input_size = 384
classifier = hub.load("https://tfhub.dev/google/imagenet/efficientnet_v2_imagenet21k_s/classification/2")
classifier2 = hub.load("https://tfhub.dev/google/imagenet/efficientnet_v2_imagenet21k_ft1k_s/classification/2")

labels_path = tf.keras.utils.get_file("imagenet21k_wordnet_lemmas.txt", "https://storage.googleapis.com/bit_models/imagenet21k_wordnet_lemmas.txt")
classes = np.array(open(labels_path).read().splitlines())
labels_path2 = tf.keras.utils.get_file("ImageNetLabels.txt", "https://storage.googleapis.com/download.tensorflow.org/data/ImageNetLabels.txt")
classes2 = np.array(open(labels_path2).read().splitlines())[1:]  # pirma eilutė – „background“
print(f"Modelis 1: {len(classes)} klasių, modelis 2: {len(classes2)} klasių")

In [ ]:
#@title 🏃 STEP 3: Pasirinkite nuotrauką { run: "auto" }
image_name = "Sporto salė" # @param ['Pritūpimai su štanga', 'Hanteliai ant stovo', 'Sporto salė', 'Svarstis (kettlebell)', 'Maratonas', 'Dviračių lenktynės', 'Krepšinis', 'Futbolas', 'Plaukimas', 'Laipiojimo siena', 'Tenisas']

image, original_image = load_image(SPORT_PHOTOS[image_name], input_size)
show_image(image, image_name)
HTML("<h1 style='font-size: 22px; color: blue;'>Modelis 1 (ImageNet-21k)<br><br><i>" + format_answer(top_classes(image, classifier, classes))
     + "</i></h1><br><h1 style='font-size: 22px; color: green;'>Modelis 2 (ImageNet-21k → 1k)<br><br><i>"
     + format_answer(top_classes(image, classifier2, classes2)) + "</i></h1>")

In [ ]:
#@title ✂️ STEP 4: Nuotraukos fragmento klasifikavimas { run: "auto" }
x = 30 # @param {type:"slider", min:0, max:100, step:1}
y = 40 # @param {type:"slider", min:0, max:100, step:1}
cut_size_in_pixels = 400 # @param {type:"integer"}

x_cord = int(original_image.shape[2] * x / 100)
y_cord = int(original_image.shape[1] * y / 100)
image_cut = original_image[:, y_cord:y_cord + cut_size_in_pixels, x_cord:x_cord + cut_size_in_pixels, :]
image_zoom = tf.image.resize(image_cut, [input_size, input_size])
show_image(image_zoom, f"Iškirpta nuo ({x_cord}, {y_cord}), dydis {image_cut.shape[1]}×{image_cut.shape[2]}")
HTML("<h1 style='font-size: 22px; color: blue;'>Modelis 1<br><br><i>" + format_answer(top_classes(image_zoom, classifier, classes))
     + "</i></h1><br><h1 style='font-size: 22px; color: green;'>Modelis 2<br><br><i>"
     + format_answer(top_classes(image_zoom, classifier2, classes2)) + "</i></h1>")

In [ ]:
#@title 📦 STEP 5: Objektų aptikimo modelis (SSD MobileNet v2, Open Images v4)
detector = hub.load("https://tfhub.dev/google/openimages_v4/ssd/mobilenet_v2/1").signatures["default"]

def draw_boxes(image, boxes, class_names, scores, max_boxes=15, min_score=0.2):
    """Nubraižo rėmelius su klasės pavadinimu ir tikimybe (supaprastinta originalo funkcija)."""
    colors = list(ImageColor.colormap.values())
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSansNarrow-Regular.ttf", 22)
    except IOError:
        font = ImageFont.load_default()
    image_pil = Image.fromarray(np.uint8(image)).convert("RGB")
    draw = ImageDraw.Draw(image_pil)
    w, h = image_pil.size
    for i in range(min(boxes.shape[0], max_boxes)):
        if scores[i] < min_score:
            continue
        ymin, xmin, ymax, xmax = boxes[i]
        color = colors[hash(class_names[i]) % len(colors)]
        left, right, top, bottom = xmin * w, xmax * w, ymin * h, ymax * h
        draw.rectangle([left, top, right, bottom], outline=color, width=4)
        label = f"{class_names[i].decode('ascii')}: {int(100 * scores[i])}%"
        text_w, text_h = font.getbbox(label)[2:]
        draw.rectangle([left, max(0, top - text_h - 4), left + text_w + 4, max(text_h + 4, top)], fill=color)
        draw.text((left + 2, max(0, top - text_h - 4)), label, fill="black", font=font)
    return np.array(image_pil)

def run_detector(p_img, show=True):
    result = {key: value.numpy() for key, value in detector(p_img).items()}
    if show:
        print(f"Rasta objektų: {len(result['detection_scores'])}")
        plt.figure(figsize=(16, 12)); plt.grid(False)
        plt.imshow(draw_boxes((p_img[0].numpy() * 255).astype("uint8"), result["detection_boxes"],
                              result["detection_class_entities"], result["detection_scores"]))
        plt.axis("off"); plt.show()
    return result

detector_output = run_detector(original_image)

In [ ]:
#@title 🎨 STEP 6: Semantinė segmentacija (HRNet, ADE20K – 150 klasių)
segmentation_model = hub.load("https://tfhub.dev/google/HRNet/ade20k-hrnetv2-w48/1")
seg_labels_path = tf.keras.utils.get_file("ade20k-id2label.json",
                                          "https://huggingface.co/datasets/huggingface/label-files/raw/main/ade20k-id2label.json")
with open(seg_labels_path) as f:
    classes_segmentation = ["background"] + list(json.load(f).values())
seg_name = lambda c: classes_segmentation[c] if c < len(classes_segmentation) else f"klasė {c}"

predictions = segmentation_model.predict(image)[0].numpy()  # (aukštis, plotis, 151 klasė)
class_map = np.argmax(predictions, axis=-1)
areas = np.bincount(class_map.ravel(), minlength=predictions.shape[-1]) / class_map.size
top4 = np.argsort(areas)[::-1][:4]  # originale indeksai įrašyti ranka; čia – 4 didžiausią plotą užimančios klasės

plt.figure(figsize=(16, 16))
for n, cls in enumerate(top4):
    plt.subplot(2, 2, n + 1); plt.xticks([]); plt.yticks([])
    plt.imshow(image[0]); plt.imshow(predictions[:, :, cls], alpha=0.5)
    plt.xlabel(f"{seg_name(cls)} – {areas[cls]:.0%} vaizdo", fontsize=14)
plt.show()

## 7 žingsnis – visos sporto nuotraukos vienoje lentelėje
Kiekvienai nuotraukai: abiejų klasifikatorių pirmasis atsakymas, užtikrinčiausiai aptikti objektai ir didžiausią plotą užimanti segmentacijos klasė.

In [ ]:
#@title 📋 STEP 7: Suvestinė visoms nuotraukoms
rows = []
for name, url in SPORT_PHOTOS.items():
    img, img_raw = load_image(url, input_size)
    (c1, p1), = top_classes(img, classifier, classes, k=1)
    (c2, p2), = top_classes(img, classifier2, classes2, k=1)
    det = run_detector(img_raw, show=False)
    seen = []
    for entity, score in zip(det["detection_class_entities"], det["detection_scores"]):
        entity = entity.decode("ascii")
        if score >= 0.3 and entity not in seen:
            seen.append(entity)
    seg = np.argmax(segmentation_model.predict(img)[0].numpy(), axis=-1)
    rows.append({"Nuotrauka": name, "Modelis 1 (21k)": f"{c1} ({p1:.2f})", "Modelis 2 (1k)": f"{c2} ({p2:.2f})",
                 "Aptikti objektai (p ≥ 0,3)": ", ".join(seen[:5]) or "–",
                 "Segmentacija: didžiausia klasė": seg_name(np.bincount(seg.ravel()).argmax())})
summary = pd.DataFrame(rows).set_index("Nuotrauka")
pd.set_option("display.max_colwidth", 80)
display(summary)

## Išvados
*(užpildoma paleidus notebook'ą)*